# FactIndic: paired factuality-aware prompting pilot

Run this notebook in **Google Colab with a GPU runtime**. It compares two prompts using the same instruction model, fixed sampling, and identical examples. It also reads the existing mT5 predictions as a baseline. The output is a draft for human review; automatic NLI/heuristic signals are not gold labels.

Upload these three files when prompted:
1. `results/baseline_hindi_test_n500_seed42/predictions.jsonl`
2. `results/baseline_codemixed_hindi_test_n500_seed42_p100/predictions.jsonl`
3. `mutant_authentic_predictions.jsonl`

This pilot samples 30 paired Hindi/synthetic examples plus 30 authentic candidates, then generates two LLM summaries per source (180 generations).

In [ ]:
!pip -q install "transformers>=4.51.0" accelerate sentencepiece


In [ ]:
from google.colab import files
from pathlib import Path
import json, random, hashlib, csv, zipfile, time, torch, re, shutil
from transformers import AutoTokenizer, AutoModelForCausalLM

uploaded = files.upload()
required = {"hindi_baseline.jsonl", "synthetic_p100.jsonl", "mutant_authentic.jsonl"}
resolved = {}
for uploaded_name in uploaded:
    normalized = re.sub(r" \(\d+\)(?=\.jsonl$)", "", uploaded_name)
    if normalized in required and normalized not in resolved:
        resolved[normalized] = uploaded_name
missing = required - set(resolved)
if missing:
    raise ValueError(f"Missing files: {sorted(missing)}. Upload the corresponding renamed copies.")
for canonical, uploaded_name in resolved.items():
    src, dst = Path(uploaded_name), Path(canonical)
    if src != dst:
        shutil.copy2(src, dst)
print("Ready:", sorted(required))


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"
SEED = 42
N_PAIRED = 30
N_AUTHENTIC = 30
MAX_SOURCE_TOKENS = 512  # cap source text for a comparable, documented input budget
MAX_NEW_TOKENS = 256
OUT = Path("/content/factindic_pilot")
OUT.mkdir(exist_ok=True)
assert torch.cuda.is_available(), "Select a Colab GPU runtime before continuing."
print("GPU:", torch.cuda.get_device_name(0))

def read_jsonl(name):
    with open(name, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

hindi = read_jsonl("hindi_baseline.jsonl")
mixed = read_jsonl("synthetic_p100.jsonl")
auth = read_jsonl("mutant_authentic.jsonl")
print(len(hindi), len(mixed), len(auth))


In [ ]:
# Pair Hindi and synthetic conditions by article ID, then sample deterministically.
h_by_id = {str(r["id"]): r for r in hindi}
m_by_id = {str(r["id"]): r for r in mixed}
paired_ids = sorted(set(h_by_id) & set(m_by_id))
if len(paired_ids) < N_PAIRED:
    raise ValueError(f"Only {len(paired_ids)} shared IDs; need {N_PAIRED}.")
rng = random.Random(SEED)
selected_ids = sorted(rng.sample(paired_ids, N_PAIRED))

# Balance authentic sample across available subsets, reproducibly.
by_subset = {}
for r in auth:
    by_subset.setdefault(str(r.get("subset", "unknown")), []).append(r)
for rows in by_subset.values():
    rows.sort(key=lambda r: str(r["id"]))
subsets = sorted(by_subset)
auth_selected = []
while len(auth_selected) < min(N_AUTHENTIC, len(auth)):
    changed = False
    for s in subsets:
        if by_subset[s] and len(auth_selected) < min(N_AUTHENTIC, len(auth)):
            auth_selected.append(by_subset[s].pop(rng.randrange(len(by_subset[s]))))
            changed = True
    if not changed: break

examples = []
for i in selected_ids:
    for condition, row, source_key in [("hindi", h_by_id[i], "text"), ("synthetic_p100", m_by_id[i], "text")]:
        examples.append({"condition":condition, "id":i, "subset":"", "source":row[source_key], "baseline_prediction":row.get("prediction", ""), "baseline_truncated":row.get("truncated", "")})
for row in auth_selected:
    examples.append({"condition":"authentic_mutant", "id":str(row["id"]), "subset":str(row.get("subset", "")), "source":row["text"], "baseline_prediction":row.get("prediction", ""), "baseline_truncated":row.get("truncated", "")})
print("Examples:", len(examples), "by condition:", {c:sum(x["condition"]==c for x in examples) for c in sorted(set(x["condition"] for x in examples))})


In [ ]:
# Load once on the Colab GPU. No local Mac model inference is used.
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto").eval()
print("Loaded", MODEL_ID)


In [ ]:
PROMPTS = {
    "standard": (
        "Summarize the source in clear, concise Hindi in 2-3 sentences (about 25-35 words). "
        "Preserve the main event and key details. Return only the summary."
    ),
    "factuality_aware": (
        "Summarize the source in clear, concise Hindi in 2-3 sentences (about 25-35 words). "
        "Use only information stated in the source. Preserve who did what to whom, and copy "
        "names, numbers, dates, negation, and attribution carefully. Do not infer or add details. "
        "Omit unclear details. Return only the summary."
    ),
}

def trim_source(source):
    ids = tok(source, add_special_tokens=False)["input_ids"]
    return tok.decode(ids[:MAX_SOURCE_TOKENS], skip_special_tokens=True), len(ids)

def generate(prompt, source):
    user_text = prompt + "\n\nSource:\n" + source
    messages = [{"role": "user", "content": user_text}]
    rendered = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    encoded = tok(rendered, return_tensors="pt")
    encoded = {key: value.to(model.device) for key, value in encoded.items()}
    prompt_length = encoded["input_ids"].shape[1]
    with torch.inference_mode():
        output = model.generate(
            **encoded,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            use_cache=True,
            pad_token_id=tok.eos_token_id,
        )
    generated_ids = output[0, prompt_length:]
    answer = tok.decode(generated_ids, skip_special_tokens=True).strip()
    generated_tokens = int(generated_ids.shape[0])
    hit_token_cap = generated_tokens >= MAX_NEW_TOKENS
    return answer, generated_tokens, hit_token_cap

rows = []
start = time.time()
for idx, ex in enumerate(examples, 1):
    visible_source, original_tokens = trim_source(ex["source"])
    for arm, prompt in PROMPTS.items():
        pred, generated_tokens, hit_token_cap = generate(prompt, visible_source)
        rows.append({
            **ex,
            "arm": arm,
            "source_visible": visible_source,
            "source_tokens_model_tokenizer": original_tokens,
            "source_was_capped": original_tokens > MAX_SOURCE_TOKENS,
            "prediction": pred,
            "generated_tokens": generated_tokens,
            "hit_new_token_cap": hit_token_cap,
        })
    if idx % 10 == 0 or idx == len(examples):
        print(f"{idx}/{len(examples)} examples; {len(rows)} generations")


In [ ]:
# Quality gate: inspect generation completeness before opening/filling human_review.csv.
required_audit_fields = {"generated_tokens", "hit_new_token_cap"}
if not rows:
    raise ValueError("No generated rows found. Run the generation cell first.")
missing_audit_fields = required_audit_fields - set(rows[0])
if missing_audit_fields:
    raise RuntimeError(
        f"This looks like an older generation cell; missing {sorted(missing_audit_fields)}. "
        "Replace and rerun the generation cell before auditing or exporting."
    )
terminal_marks = ("।", ".", "!", "?", "…", "”", '"')
cap_hits = sum(bool(r.get("hit_new_token_cap")) for r in rows)
replacement_rows = sum("�" in r.get("prediction", "") for r in rows)
replacement_at_end = sum(r.get("prediction", "").rstrip().endswith("�") for r in rows)
empty_rows = sum(not r.get("prediction", "").strip() for r in rows)
without_terminal_mark = sum(not r.get("prediction", "").rstrip().endswith(terminal_marks) for r in rows)
print(
    f"Rows: {len(rows)} | token-cap hits: {cap_hits} | empty: {empty_rows} | "
    f"replacement char: {replacement_rows} (at end: {replacement_at_end}) | "
    f"without terminal punctuation: {without_terminal_mark}"
)
if cap_hits or replacement_rows or empty_rows:
    print("STOP: Do not export for annotation. Fix/rerun generation, then rerun this audit.")
else:
    print("Mechanical checks passed. Inspect several summaries for coherent, complete sentences before annotation.")


In [ ]:
# Save generations, a blinded human annotation sheet, and provenance metadata.
# Block export if the generation audit failed or used a stale generation cell.
required_audit_fields = {"generated_tokens", "hit_new_token_cap"}
if not rows or required_audit_fields - set(rows[0]):
    raise RuntimeError("Run the current generation cell and quality-audit cell before exporting.")
if cap_hits or replacement_rows or empty_rows:
    raise RuntimeError(
        f"Export blocked: cap_hits={cap_hits}, replacement_rows={replacement_rows}, "
        f"empty_rows={empty_rows}. Fix/rerun generation before export."
    )
fields = list(rows[0].keys())
with open(OUT / "generations.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(rows)
review_fields = ["condition","id","subset","arm","source_visible","prediction","human_factuality","error_type","evidence_or_error_span","annotator_id","notes"]
with open(OUT / "human_review.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.DictWriter(f, fieldnames=review_fields); w.writeheader()
    for r in rows: w.writerow({k:r.get(k, "") for k in review_fields})
metadata = {"model":MODEL_ID,"seed":SEED,"n_paired_hindi_synthetic":len(selected_ids),"n_authentic":len(auth_selected),"prompt_arms":PROMPTS,"max_source_tokens":MAX_SOURCE_TOKENS,"max_new_tokens":MAX_NEW_TOKENS,"generation":"greedy (do_sample=False)","elapsed_seconds":round(time.time()-start,2),"generated_token_cap_hits":cap_hits,"replacement_char_rows":replacement_rows,"replacement_char_at_end_rows":replacement_at_end,"empty_generation_rows":empty_rows,"without_terminal_punctuation_rows":without_terminal_mark,"selected_paired_ids":selected_ids,"selected_authentic_ids":[r["id"] for r in auth_selected],"authentic_subset_counts":{s:sum(x["subset"]==s for x in examples if x["condition"]=="authentic_mutant") for s in subsets},"warning":"Automatic checks are quality filters only; human annotations are required for factuality results."}
(OUT / "run_metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
with zipfile.ZipFile("/content/factindic_pilot_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir(): z.write(p, arcname=p.name)
print("Saved:", "/content/factindic_pilot_results.zip")
files.download("/content/factindic_pilot_results.zip")


## After the run

1. Upload the ZIP output into the project `results/` area and keep `run_metadata.json` with the CSVs.
2. Have two humans label the same subset independently if feasible; preserve both labels and adjudicate disagreements separately. Use the protocol in `docs/factuality_annotation_protocol.md`.
3. Only after human labels exist, calculate supported/minor/major/unverifiable counts by condition and prompt arm. Keep NLI as a separate screening column.
4. The Hindi and synthetic rows are paired by source ID. The authentic MUTANT examples are a separate, unpaired condition. This is a pilot, not a broad benchmark claim.